**Funciones dinámicas**

In [0]:
from pyspark.sql import functions as F

CATALOG = "asbdevsuperstore"
BRONZE_SCHEMA = "bronze_schema"
SILVER_SCHEMA = "silver_schema"

BRONZE_TABLE = "superstore_bronze"
SILVER_TABLE = "superstore_silver"

BASE = "abfss://superstoredata@adsldevsuperstore.dfs.core.windows.net"
SILVER_PATH = f"{BASE}/silver/{SILVER_TABLE}"

**Usamos el Catalogo y Schema**

In [0]:
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SILVER_SCHEMA}")

**Limpieza**

In [0]:
df = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.{BRONZE_TABLE}")

# ============================================================
# 1) Parse de fechas (tu CSV está en MM/dd/yyyy)
# ============================================================

df = (
    df.withColumn("order_date", F.to_date(F.col("order_date"), "M/d/yyyy"))
      .withColumn("ship_date",  F.to_date(F.col("ship_date"),  "M/d/yyyy"))
)

# ============================================================
# 2) Limpieza de columnas numéricas
# ============================================================

# quantity a veces viene como texto tipo "1040 sheets"
df = df.withColumn(
    "quantity",
    F.regexp_extract(F.col("quantity").cast("string"), r"(\d+)", 1).cast("int")
)

# sales / profit / discount pueden venir con valores malformados
df = (
    df.withColumn("sales", F.expr("try_cast(sales as double)"))
      .withColumn("profit", F.expr("try_cast(profit as double)"))
      .withColumn("discount", F.expr("try_cast(discount as double)"))
)

# ============================================================
# 3) Reglas básicas de calidad (simple)
# ============================================================

df = (
    df.filter(F.col("order_id").isNotNull())
      .filter(F.col("order_date").isNotNull())
      .filter(F.col("sales").isNotNull())
      .withColumn("_silver_ts", F.current_timestamp())
)

# ============================================================
# 4) Dedupe básico (en superstore, row_id es único normalmente)
# ============================================================

if "row_id" in df.columns:
    df_silver = df.dropDuplicates(["row_id"])
else:
    df_silver = df.dropDuplicates()

# ============================================================
# 5) Guardar en Silver (Delta)
# ============================================================

(df_silver.write.format("delta")
   .mode("overwrite")
   .option("overwriteSchema", "true")
   .save(SILVER_PATH))

# ============================================================
# 6) Registrar tabla en Unity Catalog
# ============================================================

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}.{SILVER_TABLE}
USING DELTA
LOCATION '{SILVER_PATH}'
""")

# ============================================================
# 7) Validación
# ============================================================

spark.sql(f"""
SELECT COUNT(*) AS filas
FROM {CATALOG}.{SILVER_SCHEMA}.{SILVER_TABLE}
""").show()

display(
    spark.table(f"{CATALOG}.{SILVER_SCHEMA}.{SILVER_TABLE}").limit(10)
)